In [ ]:
# Today is the last day of week 13 where we do an Audit on the entire pipeline from days 8-12

"""Today, we stop building and start measuring. Im going to run my full router system
built across days 8 - 12, through apple 10-K questions and score every single result against
two criteria: did the system give up correctly when it should have (appropriate abstention),
and did it push through and solve correctly when it should have (appropriate persistence).
The goal is an honest scorecard of where my Week 6 system stands before I move into langGraph next
week.

Automated Audit Runner
Audit Dataset: Hand-crafting 20 qns accross four categories. Answerable Simple(5),
Answerable Complex(5), Unanswerable from 10-K but answerable from web(5), and Completely
unanswerable by anyone(5). Each question needs a ground truth label so you can score the
system objectively.

Automated Scorer: Write a function that runs each question through run_router, captures the result
then compares the system's behaviour against the ground truth label-> did it route correctly,
did it answer or abstain correctly, and if it answered, was the answer faithful.

Failure Classifier: For every failure, automatically tag it as one of four failure types
-> wrong route, wrong abstention, hallucinated answer, or correct route but incomplete answer.
This taxonomy will feed directly into Month 3's evaluation framework.

Audit Report Generator: At the end of the 20 runs, auto-generate a markdown report 
with the full scorecard table, failure breakdown, and a one-paragraph system summary.

"""

# Importing necessary libraries


import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)



c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_20728\3617723357.py:57: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)


In [2]:
# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [3]:
def search_local_docs(query:str) -> str:
    """Searches my local Knowledge base containing historical Apple 10-K financial documents
    (covering fiscal years up to 2024). Use this to retrieve historical sales, net revenue,
    and internal corporate performance figures.
    
    CRITICAL: Do not pass comparative or converstional questions here.
    Convert queries into strict financial line items, such as:
    - 'Apple consolidated statements of operations net sales' 
    - 'Apple total net slaes 2023 -2024' 
    - 'Summary of operations data'
    """

    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)
    return "\n\n".join([doc.node.get_content() for doc in results])


def get_doc_years(_: str = "") -> str:
    """Returns the list of available years in the 10k pdf"""
    return "Available years: 2021, 2022, 2023"

def calculator(expression: str) -> str:
    """Evaluates a basic math expression"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"
    
TOOLS = {
    "search_local_docs": search_local_docs,
    "get_doc_years": get_doc_years,
    "calculator": calculator,
}


In [4]:
# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        

In [5]:
# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"

In [6]:
# ANSWER GENERATOR

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ],

        temperature= 0,
    )

    return response.choices[0].message.content

In [7]:
# THE REVIEWER LLM

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS meaans the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-120b',
        messages = [
            {"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
            {"role": "user", "content":(
                f"Question: {question}\n\n"
                f"Source Context:\n{context}\n\n"
                f"Generated Answer:\n{answer}"

            )}
        ],
        temperature= 0
    )

    raw = response.choices[0].message.content
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason


In [8]:
# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final evrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


In [9]:
# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()

In [10]:
# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

In [11]:
# THE ROUTER CLASSIFIER

ROUTER_SYSTEM_PROMPT = """You are a query complexity classifier for a financial RAG system.

Classify the user's question as one of:
- SIMPLE: a single factual lookup requiring one retrieval and one answer
(e.g. "What was Apple's net income in FY2024?")
- COMPLEX: requires multiple steps, comparisons, calculations, or chaining
(e.g. "Compare iPhone revenue across FY2023 and FY2024 and calculate the growth rate")
- UNKNOWN: cannot be answered from a financial document at all 
(e.g. "What is the weather in Cupertino today?")

Respond in EXACTLY this format:
Classification: <SIMPLE, COMPLEX, or UNKNOWN>
Reason: <one sentence>
"""

def classify_query(question: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages = [{"role": "user", "content": f"Question: {question}"}],
        system= ROUTER_SYSTEM_PROMPT
    )
    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not Parse reason."

    print(f"🦾 Router: {classification} - {reason}")
    return classification, reason

In [12]:
# NAIVE RAG PATH 

NAIVE_RAG_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the question uning ONLY THE context provided.
Be specific - include exact figures, percentages, and fiscal year references.
If the context does not contain the answer, say so directly.

"""

def run_naive_rag(question: str) -> dict:
    print("\n⚡ Path: NAIVE RAG")
    started_at = datetime.now().isoformat()

    # single retrieval
    docs, score = retrieve_with_confidence(question)
    context = "\n\n --- \n\n".join([doc.node.get_content() for doc in docs])

    # Single generation - no reviewer, no rewrite
    answer = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }],
        system = NAIVE_RAG_SYSTEM_PROMPT
    )

    print(f"Answer: {answer}")
    return {
        "path": "naive_rag",
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat()
    }

In [13]:
# MULTI-PATH ROUTER

def run_router(question: str) -> dict:
    global usage_tracker
    usage_tracker = TokenUsage() # reset for each question

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    classification, reason = classify_query(question)

    if classification== 'SIMPLE': 
        result = run_naive_rag(question= question)

    elif classification == "COMPLEX":
        print("\n🤖 Path: AGENTIC RAG (CRAG + Reviewer)")
        result = run_crag_pipeline(question= question)
        result['path'] = "agentic_rag"
    
    else: # UNKNOWN
        print("\n🤥 Path: UNKNOWN - cannot answer from financial documents")
        result = {
            "path": "unknown",
            "question": question,
            "answer": "This question cannot be answered using the Apple 10-K document.",
            "started_at": datetime.now().isoformat(),
            "finished_at": datetime.now().isoformat()
        }
    
    usage_tracker.report()

    result["Classification"] = classification
    result["classification_reason"] = reason
    result["token_usage"] = {
        "prompt_tokens": usage_tracker.prompt_tokens,
        "completion_tokens": usage_tracker.completion_tokens,
        "total_calls": usage_tracker.total_calls,
        "estimated_cost_usd": usage_tracker.cost_estimate()
    }

    filename = f"traces/day12_{classification.lower()}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent = 2)
    print(f"\n📀 Saved to {filename}")

    return result

In [14]:
# THE 20 QN AUDIT DATASET

AUDIT_DATASET = [

    # Category 1: Answerable Simple (5 Questions) ---
    # Expected behaviour: router says SIMPLE, Naive RAG answers correctly

    {
        "id": "S1",
        "category": "answerable_simple",
        "question": "What was Apple's total net sales for fiscal year 2024",
        "expected_route": "SIMPLE",
        "expected behaviour": "answer",
        "ground_truth_hint": "Approximately $391 billion"
    },

    {
        "id": "S2",
        "category": "answerable_simple",
        "question": "What was Apple's net income for fiscal year 2024?",
        "expected_route": "SIMPLE",
        "expected_behaviour": "answer",
        "ground_truth_hint": "Approximately $93.7 billion"
    },

    {
        "id": "S3",
        "category": "answerable_simple",
        "question": "What was Apple's fiscal year 2024 end date?",
        "expected_route": "SIMPLE",
        "expected_behaviour": "answer",
        "ground_truth_hint": "September 28 2024"
    },

    {
        "id": "S4",
        "category": "answerable_simple",
        "question": "What was Apple's total revenue in fiscal year 2024?",
        "expected_route": "SIMPLE",
        "expected_behaviour": "answer",
        "ground_truth_hint": "Approximately $201.2 billion"
    },

    {
        "id": "S5",
        "category": "answerable_simple",
        "question": "How much did Apple spend on research and development in FY2024?",
        "expected_route": "SIMPLE",
        "expected_behaviour": "answer",
        "ground_truth_hint": "Approximately $31.4 billion"
    },


    # --- Category 2: Answerable Complex (5 Questions) ------
    # Expected behaviour: router says COMPLEX, Agentic RAG answers with multiple steps

    {
        "id": "C1",
        "category": "answerable_complex",
        "question": (
            "What were Apple's three largest product revenue categories in FY2024"
            "and what percentage of total net sales did each represent?"
        ),
        "expected_route": "COMPLEX",
        "expected_behaviour": "answer",
        "ground_truth_hint": "Services grew from ~$85.2B in FY2023 to ~$96.2B in FY2024"

    },

    {
        "id": "C2",
        "category": "answerable_complex",
        "question": (
            "Compare Apples's services revenue in FY2023 versus FY2024"
            "and calculate the year-over-year growth rate as a percentage."
        ),
        "expected_route": "COMPLEX",
        "expected_behaviour": "answer",
        "ground_truth_hint": "iPhone ~51.5%, Services ~24.6%, Mac ~7.7%"

    },

    {
        "id": "C3",
        "category": "answerable_complex",
        "question": (
            "Find Apple's total operating expenses and total net sales for FY2024"
            "then calculate the operating profit margin."
        ),
        "expected_route": "COMPLEX",
        "expected_behaviour": "answer",
        "ground_truth_hint": "Operating income ~$123.2B, margin ~31.5%"

    },

    {
        "id": "C4",
        "category": "answerable_complex",
        "question": (
            "What was Apple's cash and cash equivalents at the end of FY2024,"
            "what awere their total current liabilities, "
            "and what does this imoly about their short-term liquidity?"
        ),
        "expected_route": "COMPLEX",
        "expected_behaviour": "answer",
        "ground_truth_hint": "Cash ~$29.9B, current liabilities ~$176.4B"

    },

    {
        "id": "C5",
        "category": "answerable_complex",
        "question": (
            "How did Apple's gross margin percentage change between FY2023 and FY2024 "
            "and which segment - Products or Services - had a higher gross margin?"
        ),
        "expected_route": "COMPLEX",
        "expected_behaviour": "answer",
        "ground_truth_hint": "Services gross margin significantly higher then Products"

    },

    # --- Category 3: Unanswerable from 10-K but web can answer (5 questions) ---
    # Expected behaviour: CRAG fires web fallback OR router sys UNKNOWN
    {
        "id": "W1",
        "category": "web_fallback",
        "question": "What is Apple's current share price as of today?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "web_fallback_or_abstain",
        "ground_truth_hint": "Real-time data not in 10-K"
    },

    {
        "id": "W2",
        "category": "web_fallback",
        "question": "Has Apple announced any new products in 2026 so far?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "web_fallback_or_abstain",
        "ground_truth_hint": "Post-filing event not in 10-K"
    },

    {
        "id": "W3",
        "category": "web_fallback",
        "question": "What is Tim Cook's current annual salary in 2026?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "web_fallback_or_abstain",
        "ground_truth_hint": "Post-filing data not in 10-K"
    },

    {
        "id": "W4",
        "category": "web_fallback",
        "question": "What is Apple's current market capitalisation?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "web_fallback_or_abstain",
        "ground_truth_hint": "Real-time market data not in 10-K"
    },

    {
        "id": "W5",
        "category": "web_fallback",
        "question": "What did Apple announce at WWDC 2026?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "web_fallback_or_abstain",
        "ground_truth_hint": "Post_filing event not in 10-K"
    },


    # ----- Categorry 4: Completely unsanswerable (5 questions) ----
    # Expected behaviour: system abstains cleanly regardless of path
    {
        "id": "U1",
        "category": "unanswerable",
        "question": "What is the recipe for Apple's employee cafeteria lasagne?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "abstain",
        "ground_truth_hint": "Nonsense question - no source can answer this"
    },
    {
        "id": "U2",
        "category": "unanswerable",
        "question": "What colour is Tim Cook's favuorite car??",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "abstain",
        "ground_truth_hint": "Unkowable personal detail"
    },
    {
        "id": "U3",
        "category": "unanswerable",
        "question": "Predict Apple's exact net income for fiscal year 2027.",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "abstain",
        "ground_truth_hint": "Future prediction - no factual answer exists"
    },
    {
        "id": "U4",
        "category": "unanswerable",
        "question": "What is the molecular weight of the glass used in iPhone 16 screens?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "abstain",
        "ground_truth_hint": "Proprietary manufacturing detail not disclosed anywhere"
    },
    {
        "id": "U5",
        "category": "unanswerable",
        "question": "What is Tim Cook's favorite food?",
        "expected_route": "UNKNOWN",
        "expected_behaviour": "abstain",
        "ground_truth_hint": "Personal data - unknowable"
    },






]

In [15]:
# FAILURE CLASSIFIER 

FAILURE_TYPES = [
    "wrong_route",            # Router sent to wrong path
    "wrong_abstention",       # Should have answered but abstained
    "hallucinated_answer",    # Answered but with unsupported claims
    "incomplete_answer",      # Correct route, partially correct answer
    "correct",                # Everything worked as expected
]

def classify_failure(
        expected_behaviour: str,
        actual_behaviour: str,
        expected_route: str,
        actual_route: str,
        reviewer_verdict: str = "N/A"
) -> str:
    
    if actual_route != expected_route and expected_route != "UNKNOWN":
        return "wrong_route"
    if expected_behaviour == "abstain" and actual_behaviour == "answer":
        return "hallucinated answer"
    if expected_behaviour == "answer" and actual_behaviour == "abstain":
        return "wrong_abstention"
    if reviewer_verdict == "FAIL":
        return "incomplete_answer"
    return "correct"


    

In [21]:
# THE AUDIT RUNNER

def run_audit(dataset: list) -> list:
    audit_results = []

    for i, item in enumerate(dataset, 1):
        print(f"\n{'#'*60}")
        print(f"Audit Question {i}/20 [{item['id']}] - {item['category'].upper()}")
        print(f"{'#'*60}")

        try:
            result = run_router(item["question"])

            actual_route = result.get("classification", "UNKNOWN")
            final_answer = result.get("final_answer", "")
            final_verdict = result.get("final_verdict", "n/A")

            # Determine actual behaviour
            abstain_phrases = [
                "cannot find", "cannot answer", "not in the",
                "no information", "does not contain", "unable to"
            ]
            did_abstain = any(
                phrase in final_answer.lower() for phrase in abstain_phrases
            )
            actual_behaviour = "abstain" if did_abstain else "answer"

            failure_type = classify_failure(
                expected_behaviour= item["expected_behaviour"],
                actual_behaviour= actual_behaviour,
                expected_route= item["expected_route"],
                actual_route = actual_route,
                reviewer_verdict= final_verdict
            )

            audit_result = {
                **item,
                "actual_route": actual_route,
                "actual_behaviour": actual_behaviour,
                "final_answer": final_answer,
                "reviewer_verdict" : final_verdict,
                "failure_type": failure_type,
                "passed": failure_type == "correct",
                "token_usage": result.get("token_usage", {})
            }

        except Exception as e:
            print(f"🫤 Exception on question {item['id']}: {e}")
            audit_result = {
                **item,
                "actual_route": "ERROR",
                "actual_behaviour": "error",
                "final_answer": f"Exception: {str(e)}",
                "reviewer_verdict": "N/A",
                "failure_type": "error",
                "passed": False,
                "token_usage" : {}
            }

        audit_results.append(audit_result)
        status = "👍 PASS" if audit_result["passed"] else f"🤥 FAIL ({audit_result['failure_type']})"
        print(f"\nResult: {status}")
    return audit_results


In [22]:
# AUDIT REPORT GENERATOR

def generate_audit_report(audit_results: list) -> str:
    total = len(audit_results)
    passed = sum(1 for r in audit_results if r ["passed"])
    failed = total - passed

    # Metrics by category
    categories = ["answerable_simple", "answerable_complex", "web_fallback", "unanswerable"]
    category_scores = {}
    for cat in categories:
        cat_results = [r for r in audit_results if r["category"] == cat]
        cat_passed = sum(1 for r in cat_results if r["passed"])
        category_scores[cat] = (cat_passed, len(cat_results))

    # failure breakdown
    failure_counts = {}
    for r in audit_results:
        ft = r ["failure_type"]
        failure_counts[ft] = failure_counts.get(ft, 0) + 1

    # Routing accuracy
    routing_correct = sum(
        1 for r in audit_results
        if r.get("path") == r.get("expected_route")
        or r.get("expected_route") == "UNKNOWN"
    )

    # Token totals
    total_tokens = sum(
        r["token_usage"].get("prompt_tokens", 0) +
        r["token_usage"].get("completion_tokens", 0)
        for r in audit_results
    )

    total_cost = sum(
        r["token_usage"].get("estimated_cost_usd", 0.0)
        for r in audit_results
    )

    report_lines = [
       '# Month 2 Week 6 Audit Report',
       f"Generated: {datetime.now().isoformat()}",
       "",
       "## Overall Scorecard",
       f"| Metric | Score |",
       f"|-----------|-----------|",
       f" Overall pass rate | {passed}/{total} ({round(passed/total*100)}%) |",
       f"| Routing accuracy | {routing_correct}/{total} ({round(routing_correct/total*100)}%) |"
       f"| Total tokens used | {total_tokens:,} |"
       f" Estimated cost (GPT - 4o pricing) | ${total_cost:.4f} |",
       "",

    ]
    for cat, (p, t) in category_scores.items():
        report_lines.append(f"| {cat} | {p} | {t} | {round(p/t*100)}% |")

    report_lines += [
        "",
        "## Failure Breakdown",
        f"| Failure Type | Count |",
        f"|---------------|-------|",
    ]

    for ft, count in sorted(failure_counts.items(), key = lambda x: -x[1]):
        report_lines.append(f"| {ft} | {count} |")

    report_lines += [
        "",
        "## Per-Question Results",
        f"| ID | Category | Expected | Actual Route | Behaviour | Verdict | Pass |",
        f"|----|----------| ---------|--------------|-----------|---------|------|",


    ]

    for r in audit_results:
        status = "👍" if r["passed"] else "👎"
        report_lines.append(
            f"| {r['id']} | {r['category']} | {r['expected_route']} "
            f"| {r['actual_route']} | {r['actual_behaviour']} "
            f"| {r['reviewer_verdict']} | {status} |"

        )
    
    report_lines += [
        "",
        "## System Summary",
        (
            f"The Week 6 audit ran {total} questions across four categories. "
            f"The system passed {passed} ({round(passed/total*100)}%). "
            f"The most common failure type was "
            f" '{max(failure_counts, key = failure_counts.get)}' "
            f"({failure_counts[max(failure_counts, key= failure_counts.get)]} occurrences)."
            f"Total token spend across all 20 questions was {total_tokens:,} tokens "
            f"(estimated ${total_cost:.4f} as GPT-4o pricing)."
            f"These baselines will be used as the Month 3 starting benchmark."
        )

    ]

    report = "\n".join(report_lines)
    with open("day13_audit_report.md", "w") as f:
        f.write(report)
    print("\n📀 Audit report saved to day13_audit_report.md")
    print(report)
    return report




 

In [23]:
# RUN EVERYTHING
audit_results = run_audit(AUDIT_DATASET)
all_results_path = "traces/day13_full_audit.json"
with open(all_results_path, 'w') as f:
    json.dump(audit_results, f, indent = 2)

print(f"\n📀 Full audit results saved to {all_results_path}")
generate_audit_report(audit_results= audit_results)


############################################################
Audit Question 1/20 [S1] - ANSWERABLE_SIMPLE
############################################################

Question: What was Apple's total net sales for fiscal year 2024
🦾 Router: SIMPLE - The query asks for a single factual figure (Apple's total net sales FY2024), requiring only one document lookup.

⚡ Path: NAIVE RAG
📣 Top retrieval score : 0.866 (threshold: 0.5)
Answer: Apple’s total net sales for fiscal year 2024 were **$391,035 million**.

📣 Token usage report
LLM calls : 2
Prompt tokens: 1386
Completion tokens: 156
Total tokens: 1,542
Est. cost (GPT-40 pricing): $0.005025

📀 Saved to traces/day12_simple_035239.json
🫤 Exception on question S1: 'expected_behaviour'

Result: 🤥 FAIL (error)

############################################################
Audit Question 2/20 [S2] - ANSWERABLE_SIMPLE
############################################################

Question: What was Apple's net income for fiscal year 2024?
🦾 Rout

"# Month 2 Week 6 Audit Report\nGenerated: 2026-06-29T03:56:27.809459\n\n## Overall Scorecard\n| Metric | Score |\n|-----------|-----------|\n Overall pass rate | 0/20 (0%) |\n| Routing accuracy | 10/20 (50%) || Total tokens used | 8,255 | Estimated cost (GPT - 4o pricing) | $0.0304 |\n\n| answerable_simple | 0 | 5 | 0% |\n| answerable_complex | 0 | 5 | 0% |\n| web_fallback | 0 | 5 | 0% |\n| unanswerable | 0 | 5 | 0% |\n\n## Failure Breakdown\n| Failure Type | Count |\n|---------------|-------|\n| error | 12 |\n| wrong_route | 7 |\n| hallucinated answer | 1 |\n\n## Per-Question Results\n| ID | Category | Expected | Actual Route | Behaviour | Verdict | Pass |\n|----|----------| ---------|--------------|-----------|---------|------|\n| S1 | answerable_simple | SIMPLE | ERROR | error | N/A | 👎 |\n| S2 | answerable_simple | SIMPLE | UNKNOWN | answer | n/A | 👎 |\n| S3 | answerable_simple | SIMPLE | UNKNOWN | answer | n/A | 👎 |\n| S4 | answerable_simple | SIMPLE | UNKNOWN | answer | n/A | 👎 